In [1]:
import json
import numpy as np
from typing import Dict, Any, List, Set, Tuple
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

# ----------------------------
# Config
# ----------------------------
SEMANTIC_PATH = "dataset/semantic_triplets_dbpedia.json"
SPLITS_PATH   = "dataset/splits/semantic_triplet_splits_dbpedia.json"

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
E5_MODEL_NAME     = "intfloat/e5-base-v2"
MPNET_MODEL_NAME  = "sentence-transformers/all-mpnet-base-v2"

RESULT_LIMIT = 200
JACCARD_THRESHOLD = 1.0   # <- set 1.0 for "same result set", or 0.5 for partial overlap
TOPKS = [3, 5, 10]
BATCH = 32

# ----------------------------
# Load dataset + splits
# ----------------------------
with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    split_info = json.load(f)

test_idx = split_info["valid_idx"]
print(f"Loaded {len(raw_sem)} semantic entries | Test size: {len(test_idx)}")

# ----------------------------
# SPARQL execution (cached)
# ----------------------------
endpoint = SPARQLWrapper("https://dbpedia.data.dice-research.org/sparql")
endpoint.setReturnFormat(SPARQL_JSON)

answer_cache: Dict[str, Set[str]] = {}

def normalize_query(q: str) -> str:
    return "\n".join(line.strip() for line in q.strip().splitlines())

def get_answers(query: str, limit: int = 200) -> Set[str]:
    q_norm = normalize_query(query)
    if q_norm in answer_cache:
        return answer_cache[q_norm]

    endpoint.setQuery(query + f"\nLIMIT {limit}")
    try:
        res = endpoint.query().convert()
        bindings = res.get("results", {}).get("bindings", [])
    except Exception:
        bindings = []
        res = {"head": {"vars": []}}

    if not bindings:
        answers: Set[str] = set()
    else:
        vars_ = res.get("head", {}).get("vars", [])
        answers = set()
        for b in bindings:
            vals = []
            for v in vars_:
                if v in b:
                    vals.append(b[v]["value"])
            if vals:
                answers.add("|".join(vals))

    answer_cache[q_norm] = answers
    return answers

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

# ----------------------------
# Per-item pool helpers
# ----------------------------
def get_anchor_query(item: Dict[str, Any]) -> str:
    return item.get("anchor_query") or item.get("base_query") or ""

def build_pool_queries_for_item(item: Dict[str, Any]) -> List[str]:
    pool: List[str] = []
    for p in item.get("positive_queries", []):
        q = p.get("query") if isinstance(p, dict) else p
        if q:
            pool.append(q)
    for n in item.get("negative_queries", []):
        q = n.get("query") if isinstance(n, dict) else n
        if q:
            pool.append(q)

    # de-dup preserve order
    seen = set()
    out = []
    for q in pool:
        if q not in seen:
            out.append(q)
            seen.add(q)
    return out

# ----------------------------
# Build test items + unique queries (test split only)
# ----------------------------
test_items: List[Tuple[str, List[str]]] = []
unique_queries: List[str] = []
seen_q = set()

def add_unique(q: str):
    if q and q not in seen_q:
        seen_q.add(q)
        unique_queries.append(q)

for idx in test_idx:
    item = raw_sem[idx]
    anchor_q = get_anchor_query(item)
    if not anchor_q:
        continue
    pool = build_pool_queries_for_item(item)
    if not pool:
        continue

    test_items.append((anchor_q, pool))
    add_unique(anchor_q)
    for q in pool:
        add_unique(q)

print(f"Test items with non-empty pool: {len(test_items)}")
print(f"Unique queries to execute/encode: {len(unique_queries)}")

# ----------------------------
# Precompute answers ONCE for all unique queries
# ----------------------------
print("Executing SPARQL for unique queries (cached)...")
for q in unique_queries:
    _ = get_answers(q, limit=RESULT_LIMIT)

answers_map: Dict[str, Set[str]] = {q: answer_cache[normalize_query(q)] for q in unique_queries}
print("✅ Cached answers:", len(answers_map))

# ----------------------------
# Precompute gold positives + Jaccard rankings ONCE per item
# ----------------------------
# Store:
# - pos_set: relevant queries in pool (by Jaccard threshold)
# - jaccard_sorted_pool: pool sorted by Jaccard score desc (retrieval A)

pos_sets: List[Set[str]] = []
jaccard_sorted_pools: List[List[str]] = []
kept_mask: List[bool] = []

for anchor_q, pool in test_items:
    a_res = answers_map.get(anchor_q, set())

    # compute Jaccard scores for entire pool once
    scored = []
    pos_set = set()
    for q in pool:
        s = jaccard(a_res, answers_map.get(q, set()))
        scored.append((q, s))
        if s >= JACCARD_THRESHOLD:
            pos_set.add(q)

    scored.sort(key=lambda x: x[1], reverse=True)
    jaccard_sorted_pool = [q for q, _ in scored]

    pos_sets.append(pos_set)
    jaccard_sorted_pools.append(jaccard_sorted_pool)
    kept_mask.append(len(pos_set) > 0)

kept = sum(kept_mask)
print(f"Items kept (>=1 relevant under threshold): {kept} / {len(test_items)}")

# ----------------------------
# Qwen embeddings (encode ONCE for unique queries)
# ----------------------------
qwen = SentenceTransformer(QWEN_MODEL_NAME)
print(f"\n✅ Loaded baseline SentenceTransformer: {QWEN_MODEL_NAME}")

print("Encoding unique queries with Qwen (cached)...")
qwen_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = qwen.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        qwen_emb_map[q] = e
print("✅ Cached Qwen embeddings:", len(qwen_emb_map))

# Precompute Qwen ranking per item (retrieval B)
qwen_sorted_pools: List[List[str]] = []
for (anchor_q, pool) in test_items:
    a_emb = qwen_emb_map[anchor_q]
    pool_embs = np.vstack([qwen_emb_map[q] for q in pool])
    sims = pool_embs @ a_emb
    order = np.argsort(-sims)
    qwen_sorted_pools.append([pool[i] for i in order])


# ----------------------------
# Additional retrievers
# - SBERT MiniLM (all-MiniLM-L6-v2)
# - E5 (intfloat/e5-base-v2)
# - SBERT MPNet (all-mpnet-base-v2)
# - TF-IDF (word 1-2 grams)
# ----------------------------

def build_sorted_pools_from_emb_map(
    emb_map: Dict[str, np.ndarray],
    test_items: List[Tuple[str, List[str]]]
) -> List[List[str]]:
    sorted_pools: List[List[str]] = []
    for (anchor_q, pool) in test_items:
        a_emb = emb_map[anchor_q]
        pool_embs = np.vstack([emb_map[q] for q in pool])
        sims = pool_embs @ a_emb
        order = np.argsort(-sims)
        sorted_pools.append([pool[i] for i in order])
    return sorted_pools


# ---- 1) SBERT MiniLM ----
minilm = SentenceTransformer(MINILM_MODEL_NAME)
print(f"\n✅ Loaded SentenceTransformer: {MINILM_MODEL_NAME}")

print("Encoding unique queries with MiniLM (cached)...")
minilm_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = minilm.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        minilm_emb_map[q] = e
print("✅ Cached MiniLM embeddings:", len(minilm_emb_map))

minilm_sorted_pools = build_sorted_pools_from_emb_map(minilm_emb_map, test_items)


# ---- 2) E5 base v2 ----
# Note: E5 models are commonly used with prefixes like "query: " / "passage: ".
# To avoid changing your dataset text (and per your instruction "don't change this code"),
# we encode the raw query strings as-is, same as Qwen.
# ---- 2) E5 base v2 (with recommended prefixes) ----
e5 = SentenceTransformer(E5_MODEL_NAME)
print(f"\n✅ Loaded SentenceTransformer: {E5_MODEL_NAME}")

def e5_query_text(s: str) -> str:
    return f"query: {s}"

# Encode unique queries with E5 using "query:" prefix
print("Encoding unique queries with E5 (cached, query: prefix)...")
e5_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_pref = [e5_query_text(q) for q in batch_qs]
    batch_embs = e5.encode(batch_pref, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        e5_emb_map[q] = e
print("✅ Cached E5 embeddings:", len(e5_emb_map))

e5_sorted_pools = build_sorted_pools_from_emb_map(e5_emb_map, test_items)


# ---- 3) SBERT MPNet ----
mpnet = SentenceTransformer(MPNET_MODEL_NAME)
print(f"\n✅ Loaded SentenceTransformer: {MPNET_MODEL_NAME}")

print("Encoding unique queries with MPNet (cached)...")
mpnet_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = mpnet.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        mpnet_emb_map[q] = e
print("✅ Cached MPNet embeddings:", len(mpnet_emb_map))

mpnet_sorted_pools = build_sorted_pools_from_emb_map(mpnet_emb_map, test_items)


# ---- 4) TF-IDF (word 1–2 grams) ----
# Fit once on all unique queries, then rank each pool by cosine similarity to anchor.
print("\n✅ Building TF-IDF (word 1–2 grams) vectors (cached)...")
tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
X = tfidf.fit_transform(unique_queries)          # sparse (n_unique, vocab)
X = sk_normalize(X, norm="l2", axis=1)           # cosine-ready

q_to_row = {q: i for i, q in enumerate(unique_queries)}

tfidf_sorted_pools: List[List[str]] = []
for (anchor_q, pool) in test_items:
    a_row = q_to_row[anchor_q]
    a_vec = X[a_row]                             # (1, vocab)
    pool_rows = [q_to_row[q] for q in pool]
    pool_vecs = X[pool_rows]                     # (len(pool), vocab)
    sims = (pool_vecs @ a_vec.T).toarray().ravel()
    order = np.argsort(-sims)
    tfidf_sorted_pools.append([pool[i] for i in order])

print("✅ TF-IDF vectors ready:", X.shape)

# ----------------------------
# Evaluation: average P/R/F1@k given sorted pools + pos_sets
# ----------------------------
def avg_prf_at_k(sorted_pools: List[List[str]], k: int) -> Tuple[float, float, float]:
    p_sum = r_sum = f1_sum = 0.0
    total = 0

    for sorted_pool, pos_set, keep in zip(sorted_pools, pos_sets, kept_mask):
        if not keep:
            continue  # recall undefined if no relevant items exist

        topk_set = set(sorted_pool[:k])
        inter = len(topk_set & pos_set)

        p = inter / k if k > 0 else 0.0
        r = inter / len(pos_set) if len(pos_set) > 0 else 0.0
        f1 = (2*p*r/(p+r)) if (p+r) > 0 else 0.0

        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0
    return p_sum/total, r_sum/total, f1_sum/total

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3858 semantic entries | Test size: 385
Test items with non-empty pool: 385
Unique queries to execute/encode: 6953
Executing SPARQL for unique queries (cached)...
✅ Cached answers: 6953
Items kept (>=1 relevant under threshold): 384 / 385


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 380.19it/s, Materializing param=norm.weight]                              



✅ Loaded baseline SentenceTransformer: Qwen/Qwen3-Embedding-0.6B
Encoding unique queries with Qwen (cached)...
✅ Cached Qwen embeddings: 6953


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 356.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Loaded SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Encoding unique queries with MiniLM (cached)...
✅ Cached MiniLM embeddings: 6953


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 380.97it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Loaded SentenceTransformer: intfloat/e5-base-v2
Encoding unique queries with E5 (cached, query: prefix)...
✅ Cached E5 embeddings: 6953


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 387.19it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Loaded SentenceTransformer: sentence-transformers/all-mpnet-base-v2
Encoding unique queries with MPNet (cached)...
✅ Cached MPNet embeddings: 6953

✅ Building TF-IDF (word 1–2 grams) vectors (cached)...
✅ TF-IDF vectors ready: (6953, 28269)


In [2]:
print("\nAverage Style (Precision/Recall/F1) (MiniLM retrieval, Jaccard positives, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(minilm_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")

print("\nAverage Style (Precision/Recall/F1) (E5 retrieval, Jaccard positives, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(e5_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")

print("\nAverage Style (Precision/Recall/F1) (MPNet retrieval, Jaccard positives, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(mpnet_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")

print("\nAverage Style (Precision/Recall/F1) (TF-IDF 1–2g retrieval, Jaccard positives, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(tfidf_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


Average Style (Precision/Recall/F1) (MiniLM retrieval, Jaccard positives, per-item pool)
@3  | P: 0.7179 | R: 0.4934 | F1: 0.4304
@5  | P: 0.6521 | R: 0.6015 | F1: 0.4632
@10 | P: 0.5664 | R: 0.7790 | F1: 0.5160

Average Style (Precision/Recall/F1) (E5 retrieval, Jaccard positives, per-item pool)
@3  | P: 0.7240 | R: 0.4951 | F1: 0.4305
@5  | P: 0.6526 | R: 0.5988 | F1: 0.4614
@10 | P: 0.5695 | R: 0.7814 | F1: 0.5182

Average Style (Precision/Recall/F1) (MPNet retrieval, Jaccard positives, per-item pool)
@3  | P: 0.7274 | R: 0.5015 | F1: 0.4364
@5  | P: 0.6516 | R: 0.6018 | F1: 0.4631
@10 | P: 0.5693 | R: 0.7827 | F1: 0.5190

Average Style (Precision/Recall/F1) (TF-IDF 1–2g retrieval, Jaccard positives, per-item pool)
@3  | P: 0.7517 | R: 0.5116 | F1: 0.4503
@5  | P: 0.6865 | R: 0.6178 | F1: 0.4843
@10 | P: 0.5987 | R: 0.7977 | F1: 0.5402


In [3]:
print("Executed/cached unique queries:", len(answer_cache))

print("\nAverage Style (Precision/Recall/F1) (Qwen retrieval, Jaccard positives, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(qwen_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


print("\nAverage Style (Precision/Recall/F1) (HybridSparql retrieval, per-item pool)")
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(jaccard_sorted_pools, k)
    print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


Executed/cached unique queries: 6953

Average Style (Precision/Recall/F1) (Qwen retrieval, Jaccard positives, per-item pool)
@3  | P: 0.7292 | R: 0.4981 | F1: 0.4360
@5  | P: 0.6562 | R: 0.6021 | F1: 0.4660
@10 | P: 0.5656 | R: 0.7787 | F1: 0.5153

Average Style (Precision/Recall/F1) (HybridSparql retrieval, per-item pool)
@3  | P: 0.8090 | R: 0.5461 | F1: 0.4860
@5  | P: 0.7255 | R: 0.6456 | F1: 0.5130
@10 | P: 0.6185 | R: 0.8165 | F1: 0.5579
